### Data Preprocessing Pipeline

In [ ]:
## 1. Loading Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import chi2_contingency

In [ ]:
## 2. Load the dataset
df = pd.read_csv('../../../data/earthquakes_data.csv')

# Display sample data
print("Sample Data:")
print(df.sample(10))
print("\n" + "="*80 + "\n")

# Basic information
print("Dataset Information:")
df.info()
print("\n" + "="*80 + "\n")

# Dataset shape
print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\n" + "="*80 + "\n")

# Column names
print("Column Names:")
print(df.columns.tolist())
print("\n" + "="*80 + "\n")

In [ ]:
## 3. Missing Value Analysis
print("Missing Value Analysis:")
print("-" * 80)

# Count missing values
missing_count = df.isnull().sum()
print("Missing Value Counts:")
print(missing_count[missing_count > 0])
print("\n")

# Calculate missing value percentage
missing_percent = round(df.isnull().sum() * 100 / len(df), 2)
print("Missing Value Percentage:")
print(missing_percent[missing_percent > 0])
print("\n" + "="*80 + "\n")

In [ ]:
## 4. Handling Missing Values
print("Handling Missing Values:")
print("-" * 80)

# Step 1: Drop columns with >30% missing values
threshold = 30
cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()
print(f"Columns dropped (>{threshold}% missing): {cols_to_drop}")
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f"Shape after dropping columns: {df.shape}\n")

# Step 2: Drop ID-like and irrelevant columns
id_like_cols = ['Unnamed: 0', 'id', 'net', 'locationSource', 'magSource', 'place', 'updated']
existing_id_cols = [col for col in id_like_cols if col in df.columns]
print(f"Dropping ID-like/irrelevant columns: {existing_id_cols}")
df.drop(columns=existing_id_cols, inplace=True, errors='ignore')
print(f"Shape after dropping ID columns: {df.shape}\n")

# Step 3: Drop rows with missing 'depth' values
initial_rows = len(df)
df = df.dropna(subset=['depth'])
print(f"Rows dropped due to missing depth: {initial_rows - len(df)}")
print(f"Shape after dropping missing depth rows: {df.shape}\n")

# Step 4: Regression-based imputation for 'rms' column
if 'rms' in df.columns and df['rms'].isnull().sum() > 0:
    print("Applying regression-based imputation for 'rms'...")
    
    # Select predictor features
    predictors = ['mag', 'depth', 'nst', 'gap', 'dmin']
    available_features = [f for f in predictors if f in df.columns]
    
    # Separate known and missing RMS data
    df_known = df[df['rms'].notnull()].dropna(subset=available_features)
    df_missing = df[df['rms'].isnull()].dropna(subset=available_features)
    
    # Train and predict if sufficient data exists
    if not df_known.empty and not df_missing.empty:
        X_train = df_known[available_features]
        y_train = df_known['rms']
        
        model = LinearRegression()
        model.fit(X_train, y_train)
        
        # Impute missing values
        X_missing = df_missing[available_features]
        df.loc[df['rms'].isnull() & df.index.isin(df_missing.index), 'rms'] = model.predict(X_missing)
        
        print(f"RMS missing values after imputation: {df['rms'].isnull().sum()}")
    else:
        print("Insufficient data for regression imputation")
else:
    print("No 'rms' column or no missing values in 'rms'")

print("\n" + "="*80 + "\n")

# Check remaining missing values
print("Remaining Missing Values:")
remaining_missing = round(df.isnull().sum() * 100 / len(df), 2)
print(remaining_missing[remaining_missing > 0])
print("\n" + "="*80 + "\n")

In [ ]:
## 5. Feature Engineering Section

print("Feature Engineering:")
print("-" * 80)

# Step 1: Temporal feature extraction from 'time' column
if 'time' in df.columns:
    print("Extracting temporal features from 'time' column...")
    
    # Convert to datetime
    df['Datetime'] = pd.to_datetime(df['time'], errors='coerce')
    
    # Extract time components
    df['Year'] = df['Datetime'].dt.year
    df['Month'] = df['Datetime'].dt.month
    df['Day'] = df['Datetime'].dt.day
    df['Hour'] = df['Datetime'].dt.hour
    df['Minute'] = df['Datetime'].dt.minute
    df['Second'] = df['Datetime'].dt.second
    df['DayOfWeek'] = df['Datetime'].dt.dayofweek  # Monday=0, Sunday=6
    
    print("Temporal features created: Year, Month, Day, Hour, Minute, Second, DayOfWeek")
    
    # Drop original time columns
    df.drop(['time', 'Datetime'], axis=1, inplace=True)
    print("Dropped original 'time' and 'Datetime' columns\n")

# Step 2: MAGNITUDE TYPE CONVERSION TO MOMENT MAGNITUDE (Mw)

print("Converting earthquake magnitudes to Moment Magnitude (Mw)...")
print("-" * 80)

def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5  # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    elif mag_type.lower() == 'ml':
        # Local Magnitude to Moment Magnitude
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type - return NaN
        return np.nan


# Apply the conversion to the DataFrame
if 'mag' in df.columns and 'magType' in df.columns:
    df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
    print(f"✓ Moment Magnitude (Mw) created from {df['magType'].nunique()} different magnitude types")
    print(f"\nMagnitude Type Distribution:")
    print(df['magType'].value_counts())
    print(f"\nSample Magnitude Conversions:")
    print(df[['mag', 'magType', 'Mw']].head(10))
    print()
else:
    print("Warning: 'mag' or 'magType' columns not found")
    print()

# Step 3: EARTHQUAKE DAMAGE POTENTIAL CALCULATION (HAZUS Methodology)

print("Calculating earthquake damage potential using HAZUS methodology...")
print("-" * 80)

def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8  # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    # damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    damage_potential = max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5)
    # Return the final damage potential score
    return damage_potential


# Apply the conversion to the DataFrame
if 'Mw' in df.columns and 'depth' in df.columns:
    df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
    print(f"✓ Damage Potential Score calculated")
    print(f"\nDamage Potential Statistics:")
    print(df['damage_potential'].describe())
    print(f"\nSample Damage Potential Calculations:")
    print(df[['Mw', 'depth', 'damage_potential']].head(10))
    print()
else:
    print("Warning: 'Mw' or 'depth' columns not found")
    print()

# Shape after feature engineering
print(f"Shape after additional feature engineering: {df.shape}")
print("\n" + "="*80 + "\n")

# Step 4: URBANITY INDICATOR CREATION

gdf_points = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

# Load urban areas shapefile
urban_areas = gpd.read_file('../../../shp/ne_110m_admin_0_countries.shp')
urban_areas = urban_areas.to_crs(gdf_points.crs)

# Spatial join to check if points lie within urban areas
gdf_joined = gpd.sjoin(gdf_points, urban_areas, how='left', predicate='within')

# Create urbanity indicator: 1 if inside urban area, else 0
gdf_joined['urbanity_indicator'] = gdf_joined['index_right'].notnull().astype(int)

# Check distribution
print(gdf_joined['urbanity_indicator'].value_counts())

# Function to compute risk score based on urbanity
def compute_risk_score(row):
  mw = row['Mw']
  depth = max(row['depth'], 1) # Prevent division by zero
  urbanity = row['urbanity_indicator']

  if urbanity == 1: # Urban area
      # Amplified formula for urban areas: higher sensitivity
      risk_score = (mw**2) * (1/depth) * 1.5
  else: # Rural Area
      # Standard formula for rural area
      risk_score = mw * (1/depth)

  # Cap score to max of 10 for normalization
  return risk_score

# Apply to compute urban risk score
gdf_joined['risk_score'] = gdf_joined.apply(compute_risk_score, axis=1)

# Step 5: RISK SCORE CATEGORIZATION

# Function to assign risk score category based on score
def assign_risk_category(score):
  if score >= 7.5:
    return 'Very High'
  elif score >=5.0:
    return 'High'
  elif score >= 2.5:
    return 'Medium'
  else:
    return 'Low'

# Apply to assign risk category
gdf_joined['risk_category'] = gdf_joined['risk_score'].apply(assign_risk_category)

# Check results
print(gdf_joined[['Mw', 'depth', 'urbanity_indicator', 'risk_score', 'risk_category']].head(10))
print("\nRisk Category Distribution:")
print(gdf_joined['risk_category'].value_counts())

In [ ]:
# Add features back to original DataFrame
df['urbanity_indicator'] = gdf_joined['urbanity_indicator']
df['risk_score'] = gdf_joined['risk_score']
df['risk_category'] = gdf_joined['risk_category']

In [ ]:
df[['damage_potential', 'urbanity_indicator', 'risk_score', 'risk_category']].head()

In [ ]:
df.isnull().sum()

In [ ]:
# Drop nulls in 'risk_score' and 'Mw'
df.dropna(subset=['risk_score', 'Mw'], inplace=True)

In [ ]:
# List of columns to drop
cols_to_drop = ['Month', 'Day', 'Hour', 'Minute', 'Second', 'DayOfWeek', 'magType', 'type', 'status', 'mag','risk_category']

# Drop columns from the dataframe
df = df.drop(columns=cols_to_drop)

# Display remaining columns to verify
print(df.columns)

In [ ]:
# # Ordinal Encoding 'risk_category' 
# risk_category_mapping = {
#     'Low': 1,
#     'Medium': 2,
#     'High': 3,
#     'Very High': 4
# }
# df['risk_category_encoded'] = df['risk_category'].map(risk_category_mapping)    

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df['Year'].unique()

In [ ]:
## Create 'decade' feature from 'Year'
df['decade'] = (df['Year'] // 10) * 10
df['decade'].unique()

In [ ]:
## drop 'Year' column as 'decade' is created
df = df.drop(columns=['Year'])

In [ ]:
## Final DataFrame Preview
df.head()

In [ ]:
## Feature Scaling
# Initialize scalers
std_scaler = StandardScaler()
min_max_scaler = MinMaxScaler()

# Standard scale continuous features
continuous_features = ['depth', 'rms', 'Mw', 'damage_potential']
df[continuous_features] = std_scaler.fit_transform(df[continuous_features])

# MinMax scale decade16
df[['decade']] = min_max_scaler.fit_transform(df[['decade']])

# urbanity_indicator remains unchanged

In [65]:
df.head()

,latitude,longitude,depth,rms,Mw,damage_potential,urbanity_indicator,risk_score,decade
16,41.758,23.249,-0.430568,0.340968,1.475494,1.340698,1,4.928040,0.0
17,41.802,23.108,-0.430568,0.303303,1.199571,1.212156,1,4.678560,0.0
18,52.763,160.277,-0.291258,0.480096,2.517872,1.261983,0,0.256667,0.0
19,51.424,161.638,-0.430568,0.441408,2.211290,1.683476,0,0.500000,0.0
20,30.684,100.608,-0.430568,0.355616,1.582798,1.390686,1,5.026810,0.0


In [66]:
## Create a dataframe without latitude and longitude for modeling
df_model = df.drop(columns=['latitude', 'longitude'])
df_model.head()

,depth,rms,Mw,damage_potential,urbanity_indicator,risk_score,decade
16,-0.430568,0.340968,1.475494,1.340698,1,4.928040,0.0
17,-0.430568,0.303303,1.199571,1.212156,1,4.678560,0.0
18,-0.291258,0.480096,2.517872,1.261983,0,0.256667,0.0
19,-0.430568,0.441408,2.211290,1.683476,0,0.500000,0.0
20,-0.430568,0.355616,1.582798,1.390686,1,5.026810,0.0


In [67]:
df_model.describe()

,depth,rms,Mw,damage_potential,urbanity_indicator,risk_score,decade
count,1.097150e+05,1.097150e+05,1.097150e+05,1.097150e+05,109715.000000,109715.000000,109715.000000
mean,-1.865163e-17,5.512592e-16,2.030955e-16,9.470881e-16,0.221556,1.090893,0.753372
std,1.000005e+00,1.000005e+00,1.000005e+00,1.000005e+00,0.415296,4.596779,0.209688
min,-6.070278e-01,-8.243420e+00,-1.620981e+00,-3.000832e+00,0.000000,0.007580,0.000000
25%,-4.770048e-01,-3.869593e-01,-1.007817e+00,-5.384142e-01,0.000000,0.154545,0.666667
50%,-2.633956e-01,2.201568e-02,1.112057e-01,2.494625e-01,0.000000,0.297691,0.750000
75%,-9.622323e-02,3.733433e-01,8.577322e-01,5.953498e-01,0.000000,0.655000,0.916667
max,5.931271e+00,1.751162e+02,5.277107e+00,2.820831e+00,1.000000,85.617038,1.000000


In [68]:
## 6. Final Dataset Overview
print("Final Dataset Overview:")
print("-" * 80)
print(df_model.info())
print("-" * 80)

## Save the preprocessed dataset
df_model.to_csv('../../../data/earthquakes_data_preprocessed.csv', index=False)

## End of Data Preprocessing Pipeline

Final Dataset Overview:
--------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 109715 entries, 16 to 110132
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   depth               109715 non-null  float64
 1   rms                 109715 non-null  float64
 2   Mw                  109715 non-null  float64
 3   damage_potential    109715 non-null  float64
 4   urbanity_indicator  109715 non-null  int32  
 5   risk_score          109715 non-null  float64
 6   decade              109715 non-null  float64
dtypes: float64(6), int32(1)
memory usage: 6.3 MB
None
--------------------------------------------------------------------------------
